<a href="https://colab.research.google.com/github/codebyhasib/Code-Project/blob/main/Feature_Engineering_and_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("All required libraries successfully imported!")


All required libraries successfully imported!


In [11]:
df=pd.read_csv("/datalab/soundwave_music_users.csv")
df

,user_id,signup_date,age,subscription_tier,primary_genre,daily_listen_mins,songs_skipped_per_day,playlists_created,device_type,churn_risk
0,SW-1001,2023-01-15,24,Free,Hip-Hop,45.0,18,1,Mobile,1
1,SW-1002,2022-11-04,38,Gold,Rock,130.0,4,8,Desktop,0
2,SW-1003,2023-03-20,29,Silver,Pop,85.0,9,4,Mobile,0
3,SW-1004,2021-08-12,45,Gold,Jazz,150.0,3,12,Smart Speaker,0
4,SW-1005,2023-05-28,19,Free,Electronic,35.0,22,0,Mobile,1
5,SW-1006,2022-07-10,32,Bronze,Pop,65.0,12,3,Mobile,0
6,SW-1007,2023-02-14,27,Free,Hip-Hop,50.0,16,1,Mobile,1
7,SW-1008,2021-12-05,52,Gold,Rock,140.0,5,10,Desktop,0
8,SW-1009,2023-06-18,34,Silver,Electronic,90.0,8,5,Desktop,0
9,SW-1010,2022-04-22,41,Bronze,Jazz,55.0,14,2,Smart Speaker,0


In [12]:
print("Dataset Shape (Rows, Columns):", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())

Dataset Shape (Rows, Columns): (40, 10)

First 5 rows of the dataset:
   user_id signup_date  age subscription_tier primary_genre  \
0  SW-1001  2023-01-15   24              Free       Hip-Hop   
1  SW-1002  2022-11-04   38              Gold          Rock   
2  SW-1003  2023-03-20   29            Silver           Pop   
3  SW-1004  2021-08-12   45              Gold          Jazz   
4  SW-1005  2023-05-28   19              Free    Electronic   

   daily_listen_mins  songs_skipped_per_day  playlists_created    device_type  \
0               45.0                     18                  1         Mobile   
1              130.0                      4                  8        Desktop   
2               85.0                      9                  4         Mobile   
3              150.0                      3                 12  Smart Speaker   
4               35.0                     22                  0         Mobile   

   churn_risk  
0           1  
1           0  
2           0  


In [13]:
print("\nChurn Risk Counts:")
print(df['churn_risk'].value_counts())


Churn Risk Counts:
churn_risk
0    30
1    10
Name: count, dtype: int64


In [19]:
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['signup_month'] = df['signup_date'].dt.month
df['signup_day_name'] = df['signup_date'].dt.day_name()
df['tenure_days'] = (pd.to_datetime('2024-01-01') - df['signup_date']).dt.days
datetime_cols = ['signup_date', 'signup_month', 'signup_day_name', 'tenure_days']
print(df[datetime_cols].head())

  signup_date  signup_month signup_day_name  tenure_days
0  2023-01-15             1          Sunday          351
1  2022-11-04            11          Friday          423
2  2023-03-20             3          Monday          287
3  2021-08-12             8        Thursday          872
4  2023-05-28             5          Sunday          218


In [20]:
songs_listened = df['daily_listen_mins'] / 3.5
df['skip_rate'] = df['songs_skipped_per_day'] / songs_listened
df['is_curator'] = (df['playlists_created'] >= 5).astype(int)
cols_to_show = ['daily_listen_mins', 'songs_skipped_per_day', 'skip_rate', 'is_curator']
print(df[cols_to_show].head())

   daily_listen_mins  songs_skipped_per_day  skip_rate  is_curator
0               45.0                     18   1.400000           0
1              130.0                      4   0.107692           1
2               85.0                      9   0.370588           0
3              150.0                      3   0.070000           1
4               35.0                     22   2.200000           0


In [21]:
tier_mapping = {'Free': 0, 'Bronze': 1, 'Silver': 2, 'Gold': 3}
df['tier_code'] = df['subscription_tier'].map(tier_mapping)
device_dummies = pd.get_dummies(df['device_type'], prefix='device', drop_first=True, dtype=int)
df = pd.concat([df, device_dummies], axis=1)
cols_to_show = ['subscription_tier', 'tier_code', 'device_type'] + list(device_dummies.columns)
print(df[cols_to_show].head())

  subscription_tier  tier_code    device_type  device_Mobile  \
0              Free          0         Mobile              1   
1              Gold          3        Desktop              0   
2            Silver          2         Mobile              1   
3              Gold          3  Smart Speaker              0   
4              Free          0         Mobile              1   

   device_Smart Speaker  
0                     0  
1                     0  
2                     0  
3                     1  
4                     0  


In [22]:
bins = [18, 25, 40, 70]
labels = ['Young (18-25)', 'Adult (26-40)', 'Mature (41+)']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, include_lowest=True)
print(df['age_group'].value_counts())

age_group
Adult (26-40)    19
Mature (41+)     13
Young (18-25)     8
Name: count, dtype: int64


In [23]:
feature_cols = ['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days', 'skip_rate', 'tier_code']
X = df[feature_cols]
y = df['churn_risk']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
train_churn_pct = y_train.mean() * 100
test_churn_pct = y_test.mean() * 100
print(f"y_train-এ Churn Risk (1)-এর হার: {train_churn_pct:.2f}%")
print(f"y_test-এ Churn Risk (1)-এর হার: {test_churn_pct:.2f}%")

y_train-এ Churn Risk (1)-এর হার: 25.00%
y_test-এ Churn Risk (1)-এর হার: 25.00%


In [24]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("X_train_scaled Means (গড় ≈ 0):")
print(X_train_scaled.mean(axis=0))
print("\nX_train_scaled Standard Deviations (মানক বিচ্যুতির পরিমাণ ≈ 1):")
print(X_train_scaled.std(axis=0))

X_train_scaled Means (গড় ≈ 0):
[-6.93889390e-18  1.38777878e-17  1.38777878e-17 -4.85722573e-17
 -4.85722573e-17]

X_train_scaled Standard Deviations (মানক বিচ্যুতির পরিমাণ ≈ 1):
[1. 1. 1. 1. 1.]


In [25]:
num_features = ['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days']
cat_features = ['primary_genre', 'device_type']
X = df[num_features + cat_features]
y = df['churn_risk']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)
model_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42))
    ]
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy Score: {accuracy:.4f}")

Test Accuracy Score: 1.0000
